# Step 5 — Comparing the three advanced models

Compares LightGBM, KGIN + features and LSTM + attention on the same searches. Nothing is retrained: each model's saved scores are loaded and ranked.

**Setup.** Same as steps 1–4: 70/15/15 split by search (seed 4222), full lists, NDCG@10 with gains 0 / 1 / 3 / 7, and ties broken at random. Run it after all three step 4 notebooks. A CPU runtime is enough.

| Section | Content | Output (`comparison/`) |
|---|---|---|
| 0 | Setup | — |
| 1 | Load the saved scores | `results.csv` |
| 2 | Paired tests on test | `paired_tests.csv` |
| 3 | Segments (validation) | `segments.csv` |

## Section 0 · Setup

- On Colab this mounts Drive and uses the project folders. Elsewhere it uses the same folders inside the notebook's own folder.
- `ndcg_lists`: NDCG@10 of each search, the same function as in the LightGBM notebook (ties at random, missing scores last, searches with no interaction dropped).
- `paired_test`: gap between two models on the same searches, with a 95% interval, a paired t-test and the share of searches each model wins.

In [1]:
import os, sys
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/BT4222_Group10_JDSearch" if IN_COLAB else "."
PROC, FEAT = f"{ROOT}/processed", f"{ROOT}/features"    # steps 1 and 2
CMP = f"{ROOT}/comparison"                              # this notebook's output
os.makedirs(CMP, exist_ok=True)

SPLITS = {"train": 0, "valid": 1, "test": 2}
GAIN = np.array([0.0, 1.0, 3.0, 7.0])                   # gain per label: 2^label - 1

def ndcg_lists(sid, label, score, k=10, seed=0):
    """NDCG@k of each search. Rank by score (high first, NaN last, ties at random); searches with no interaction are dropped."""
    d = pd.DataFrame({"sid": sid, "label": label, "score": score})
    d["tie"] = np.random.default_rng(seed).random(len(d))
    d = d.sort_values(["sid", "score", "tie"], ascending=[True, False, True], na_position="last")
    pos = d.groupby("sid").cumcount().to_numpy()                   # 0 = top
    dcg = pd.Series(np.where(pos < k, GAIN[d.label.to_numpy()] / np.log2(pos + 2), 0.0)).groupby(d.sid.to_numpy()).sum()
    best = d.sort_values(["sid", "label"], ascending=[True, False]) # ideal order
    bpos = best.groupby("sid").cumcount().to_numpy()
    idcg = pd.Series(np.where(bpos < k, GAIN[best.label.to_numpy()] / np.log2(bpos + 2), 0.0)).groupby(best.sid.to_numpy()).sum()
    ok = idcg > 0
    return dcg[ok] / idcg[ok]

def paired_test(a, b):
    """a - b on the same searches (two outputs of ndcg_lists): mean gap, 95% interval, p-value, share of searches better / worse."""
    d = a - b
    half = 1.96 * d.std() / np.sqrt(len(d))
    return {"difference": d.mean(), "95% low": d.mean() - half, "95% high": d.mean() + half,
            "p-value": stats.ttest_rel(a, b).pvalue, "share better": (d > 0).mean(), "share worse": (d < 0).mean()}

print("pandas", pd.__version__, "| Colab", IN_COLAB, "| project folder:", ROOT)

Mounted at /content/drive
pandas 2.2.3 | Colab True | project folder: /content/drive/MyDrive/BT4222_Group10_JDSearch


## Section 1 · Load the saved scores

- Each score file has one row per shown product (`sid`, `wid`, `score`) in `exposures.parquet` order. The cell stops if a file's rows don't line up.
- Popularity is step 3's baseline: the 90-day cart count from step 2.
- The table repeats each model's train, validation and test NDCG@10. It should match the step 3 and step 4 notebooks.

| File | Notebook |
|---|---|
| `lightgbm/score_tuned.parquet` | step 4a, final LightGBM |
| `model/score_kgin_features.parquet` | step 4b, KGIN + features |
| `advanced/score_lstm_attn_nolr.parquet` | step 4c, LSTM + attention (28 features) |

In [2]:
ex = pd.read_parquet(f"{PROC}/exposures.parquet", columns=["sid", "wid", "label", "split"])
SID, LAB = ex.sid.to_numpy(), ex.label.to_numpy()

FILES = {"lightgbm": "lightgbm/score_tuned.parquet",
         "kgin + features": "model/score_kgin_features.parquet",
         "lstm + attention": "advanced/score_lstm_attn_nolr.parquet"}
S = {}
for name, f in FILES.items():
    s = pd.read_parquet(f"{ROOT}/{f}")
    assert (s.sid.to_numpy() == SID).all() and (s.wid.to_numpy() == ex.wid.to_numpy()).all(), f"{f}: rows don't line up with exposures"
    S[name] = s.score.to_numpy()
S["popularity"] = pq.read_table(f"{FEAT}/section3_popularity.parquet", columns=["pop_cart_90d"])["pop_cart_90d"].to_numpy()

per_search = {}
for split, code in SPLITS.items():
    m = (ex.split == code).to_numpy()
    per_search[split] = {n: ndcg_lists(SID[m], LAB[m], s[m]) for n, s in S.items()}
results = pd.DataFrame({sp: {n: v.mean() for n, v in d.items()} for sp, d in per_search.items()})
results.to_csv(f"{CMP}/results.csv")
print("NDCG@10 by split:\n" + results.round(4).to_string())

NDCG@10 by split:
                   train   valid    test
lightgbm          0.4208  0.4065  0.4063
kgin + features   0.4161  0.4034  0.4060
lstm + attention  0.4034  0.4025  0.4033
popularity        0.3326  0.3342  0.3329


## Section 2 · Paired tests on test

Each pair of models is scored on the same 25,760 test searches. The LightGBM and KGIN notebooks already compare their own models with popularity, so this adds the LSTM against popularity and each advanced model against the others.

| Comparison (test) | Difference | 95% interval | p-value | Better / worse searches |
|---|---|---|---|---|
| LSTM + attention − popularity | +0.0704 | +0.0674 to +0.0735 | < 0.0001 | 36% / 20% |
| LightGBM − KGIN + features | +0.0003 | −0.0015 to +0.0020 | 0.78 | 22% / 22% |
| LightGBM − LSTM + attention | +0.0029 | +0.0013 to +0.0046 | 0.0004 | 21% / 20% |
| KGIN + features − LSTM + attention | +0.0027 | +0.0012 to +0.0042 | 0.0005 | 20% / 19% |

- The LSTM beats popularity by 0.0704, so all three advanced models clearly beat the best baseline.
- LightGBM and KGIN + features are tied: the gap is 0.0003 and the interval includes 0 (p = 0.78).
- Both are ahead of the LSTM by about 0.003 (p < 0.001). The gap is real but small: each wins on only about 1% more searches than it loses.

In [3]:
T = per_search["test"]
PAIRS = [("lstm + attention", "popularity"), ("lightgbm", "kgin + features"),
         ("lightgbm", "lstm + attention"), ("kgin + features", "lstm + attention")]
paired = pd.DataFrame({f"{a} - {b}": paired_test(T[a], T[b]) for a, b in PAIRS}).T
paired.to_csv(f"{CMP}/paired_tests.csv")
print(f"test searches: {len(T['lightgbm']):,}\n")
print("paired gaps on test:\n" + paired.round(4).to_string())

test searches: 25,760

paired gaps on test:
                                    difference  95% low  95% high  p-value  share better  share worse
lstm + attention - popularity           0.0704   0.0674    0.0735   0.0000        0.3568       0.2006
lightgbm - kgin + features              0.0003  -0.0015    0.0020   0.7751        0.2200       0.2194
lightgbm - lstm + attention             0.0029   0.0013    0.0046   0.0004        0.2100       0.1982
kgin + features - lstm + attention      0.0027   0.0012    0.0042   0.0005        0.2000       0.1870


## Section 3 · Segments

Validation only. Validation NDCG@10 per group of searches, with the same groups as the LightGBM notebook (Section 8):
- **cold products:** how many of the search's interacted products are in no one's history before 17 Oct (`is_cold`);
- **activity:** the user's clicks in the last 7 days (step 2, Section 4);
- **new vs repeat search:** repeat = in the last 7 days the user acted on results of a similar query (`qmatch_cate4_share_7d` is set).

| Group | Searches | LightGBM | KGIN + features | LSTM + attention | Popularity |
|---|---|---|---|---|---|
| interacted products: none cold | 17,567 | **0.4982** | 0.4930 | 0.4930 | 0.4274 |
| some cold | 3,561 | **0.2744** | 0.2694 | 0.2707 | 0.2070 |
| all cold | 4,631 | 0.1594 | **0.1664** | 0.1609 | 0.0733 |
| clicks in 7 days: 0 | 4,184 | **0.3886** | 0.3827 | 0.3840 | 0.3468 |
| 1–20 | 9,648 | **0.4131** | 0.4112 | 0.4105 | 0.3336 |
| 21–100 | 8,605 | **0.4088** | 0.4060 | 0.4051 | 0.3321 |
| over 100 | 3,322 | **0.4040** | 0.3997 | 0.3957 | 0.3206 |
| new search | 22,496 | **0.3981** | 0.3957 | 0.3948 | 0.3386 |
| repeat search | 3,263 | **0.4642** | 0.4565 | 0.4552 | 0.3032 |

- LightGBM is highest in 8 of 9 groups, but never by more than 0.008.
- When every interacted product is cold, KGIN + features is highest (0.1664). Its product vectors come from the knowledge graph, which still describes products nobody has bought before.
- All three gain most over popularity on repeat searches (+0.152 to +0.161) and least for users with no clicks in the last 7 days (+0.036 to +0.042).
- These gaps are on validation and are not tested for significance.

In [4]:
m = (ex.split == SPLITS["valid"]).to_numpy()
pos = m & (LAB > 0)
is_cold = pq.read_table(f"{FEAT}/section3_popularity.parquet", columns=["is_cold"])["is_cold"].to_numpy() == 1
cold_share = pd.Series(is_cold[pos]).groupby(SID[pos]).mean()
cold = pd.Series(np.select([cold_share == 0, cold_share == 1], ["none cold", "all cold"], "some cold"), index=cold_share.index)
ctx = pd.read_parquet(f"{FEAT}/section4_context.parquet", columns=["sid", "clicks_7d"]).set_index("sid")
activity = pd.cut(ctx.clicks_7d, [-1, 0, 20, 100, np.inf], labels=["0", "1-20", "21-100", "over 100"]).astype(str)
qm = pd.read_parquet(f"{FEAT}/section2_affinity.parquet", columns=["sid", "qmatch_cate4_share_7d"])
repeat = qm.qmatch_cate4_share_7d.notna().groupby(qm.sid).any().map({True: "repeat", False: "new"})

GROUPS = {"interacted products": (cold, ["none cold", "some cold", "all cold"]),
          "clicks in the last 7 days": (activity, ["0", "1-20", "21-100", "over 100"]),
          "search": (repeat, ["new", "repeat"])}
rows = []
for title, (group, order) in GROUPS.items():
    g_row = group.reindex(SID).to_numpy()
    for g in order:
        s = m & (g_row == g)
        row = {"grouping": title, "group": g, "searches": pd.Series(SID[s]).nunique()}
        row.update({n: ndcg_lists(SID[s], LAB[s], v[s]).mean() for n, v in S.items()})
        rows.append(row)
segments = pd.DataFrame(rows)
segments.to_csv(f"{CMP}/segments.csv", index=False)
print("validation NDCG@10 by segment:\n" + segments.round(4).to_string(index=False))

validation NDCG@10 by segment:
                 grouping     group  searches  lightgbm  kgin + features  lstm + attention  popularity
      interacted products none cold     17567    0.4982           0.4930            0.4930      0.4274
      interacted products some cold      3561    0.2744           0.2694            0.2707      0.2070
      interacted products  all cold      4631    0.1594           0.1664            0.1609      0.0733
clicks in the last 7 days         0      4184    0.3886           0.3827            0.3840      0.3468
clicks in the last 7 days      1-20      9648    0.4131           0.4112            0.4105      0.3336
clicks in the last 7 days    21-100      8605    0.4088           0.4060            0.4051      0.3321
clicks in the last 7 days  over 100      3322    0.4040           0.3997            0.3957      0.3206
                   search       new     22496    0.3981           0.3957            0.3948      0.3386
                   search    repeat      3